# 第二十四章：Vision Transformer (ViT) — PyTorch 實作

本 notebook 實作 Vision Transformer，展示如何將圖片轉換為 patch 序列並用 Transformer 處理。

**論文**: An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale  
**作者**: Alexey Dosovitskiy et al.  
**機構**: Google Research, Brain Team  
**發表**: arXiv:2010.11929, October 2020 (ICLR 2021)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import math

# 設定隨機種子
torch.manual_seed(42)
np.random.seed(42)

# 檢查 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用裝置: {device}')

## 24.1 Patch Embedding 視覺化

首先，讓我們視覺化圖片如何被分割成 patches。

In [ ]:
def visualize_patches(image, patch_size=16):
    """
    視覺化圖片的 patch 分割
    
    Args:
        image: numpy array (H, W, C) 或 (H, W)
        patch_size: patch 大小
    """
    if len(image.shape) == 2:
        image = np.stack([image] * 3, axis=-1)
    
    H, W, C = image.shape
    num_patches_h = H // patch_size
    num_patches_w = W // patch_size
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # 原始圖片
    axes[0].imshow(image, cmap='gray' if C == 1 else None)
    axes[0].set_title('Original Image', fontsize=12)
    axes[0].axis('off')
    
    # 帶 patch 網格的圖片
    axes[1].imshow(image, cmap='gray' if C == 1 else None)
    
    # 繪製網格線
    for i in range(num_patches_h + 1):
        axes[1].axhline(y=i * patch_size, color='red', linewidth=1)
    for j in range(num_patches_w + 1):
        axes[1].axvline(x=j * patch_size, color='red', linewidth=1)
    
    # 標記 patch 編號
    for i in range(num_patches_h):
        for j in range(num_patches_w):
            patch_idx = i * num_patches_w + j
            axes[1].text(
                j * patch_size + patch_size // 2,
                i * patch_size + patch_size // 2,
                str(patch_idx),
                ha='center', va='center',
                color='white', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='blue', alpha=0.5)
            )
    
    axes[1].set_title(f'Patches ({patch_size}x{patch_size}), Total: {num_patches_h * num_patches_w}', fontsize=12)
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig('patch_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'圖片大小: {H}x{W}')
    print(f'Patch 大小: {patch_size}x{patch_size}')
    print(f'Patch 數量: {num_patches_h} x {num_patches_w} = {num_patches_h * num_patches_w}')

# 建立一個簡單的測試圖片
test_image = np.random.rand(64, 64, 3)
visualize_patches(test_image, patch_size=16)

## 24.2 核心模組實作

### 24.2.1 Patch Embedding

In [ ]:
class PatchEmbedding(nn.Module):
    """
    將圖片分割成 patches 並進行線性投影
    
    使用卷積實作，等價於：
    1. 將圖片分割成 (H/P) x (W/P) 個大小為 P x P 的 patches
    2. 將每個 patch 展平成 P*P*C 維向量
    3. 通過線性層投影到 embed_dim 維
    """
    def __init__(self, img_size=224, patch_size=16, in_channels=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        
        # 使用卷積實現 patch embedding
        # kernel_size = stride = patch_size，確保不重疊
        self.proj = nn.Conv2d(
            in_channels, embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )
    
    def forward(self, x):
        """
        Args:
            x: (B, C, H, W) 輸入圖片
        Returns:
            (B, num_patches, embed_dim) patch embeddings
        """
        B, C, H, W = x.shape
        assert H == self.img_size and W == self.img_size, \
            f"輸入圖片大小 ({H}x{W}) 與預期 ({self.img_size}x{self.img_size}) 不符"
        
        # (B, C, H, W) -> (B, embed_dim, H/P, W/P)
        x = self.proj(x)
        
        # (B, embed_dim, H/P, W/P) -> (B, embed_dim, num_patches) -> (B, num_patches, embed_dim)
        x = x.flatten(2).transpose(1, 2)
        
        return x

# 測試 Patch Embedding
patch_embed = PatchEmbedding(img_size=224, patch_size=16, in_channels=3, embed_dim=768)
test_input = torch.randn(2, 3, 224, 224)
output = patch_embed(test_input)
print(f'輸入形狀: {test_input.shape}')
print(f'輸出形狀: {output.shape}')
print(f'Patch 數量: {patch_embed.num_patches}')

### 24.2.2 Multi-Head Self-Attention

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    """
    多頭自注意力機制
    
    將輸入線性投影為 Q, K, V，計算縮放點積注意力，然後投影回原維度。
    """
    def __init__(self, embed_dim, num_heads, dropout=0.0, qkv_bias=True):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5  # 1/sqrt(d_k)
        
        assert embed_dim % num_heads == 0, "embed_dim 必須能被 num_heads 整除"
        
        # Q, K, V 的線性投影，合併成一個矩陣以提高效率
        self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=qkv_bias)
        
        # 輸出投影
        self.proj = nn.Linear(embed_dim, embed_dim)
        
        # Dropout
        self.attn_dropout = nn.Dropout(dropout)
        self.proj_dropout = nn.Dropout(dropout)
    
    def forward(self, x, return_attention=False):
        """
        Args:
            x: (B, N, embed_dim) 輸入序列
            return_attention: 是否返回注意力權重
        Returns:
            output: (B, N, embed_dim)
            attention: (B, num_heads, N, N) 如果 return_attention=True
        """
        B, N, C = x.shape
        
        # 計算 Q, K, V
        # (B, N, C) -> (B, N, 3*C) -> (B, N, 3, num_heads, head_dim) -> (3, B, num_heads, N, head_dim)
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # 各為 (B, num_heads, N, head_dim)
        
        # 計算注意力分數: (B, num_heads, N, head_dim) @ (B, num_heads, head_dim, N) -> (B, num_heads, N, N)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_dropout(attn)
        
        # 加權求和: (B, num_heads, N, N) @ (B, num_heads, N, head_dim) -> (B, num_heads, N, head_dim)
        x = attn @ v
        
        # 合併多頭: (B, num_heads, N, head_dim) -> (B, N, num_heads, head_dim) -> (B, N, embed_dim)
        x = x.transpose(1, 2).reshape(B, N, C)
        
        # 輸出投影
        x = self.proj(x)
        x = self.proj_dropout(x)
        
        if return_attention:
            return x, attn
        return x

# 測試 Multi-Head Self-Attention
mhsa = MultiHeadSelfAttention(embed_dim=768, num_heads=12)
test_input = torch.randn(2, 197, 768)  # 196 patches + 1 CLS token
output, attn = mhsa(test_input, return_attention=True)
print(f'輸入形狀: {test_input.shape}')
print(f'輸出形狀: {output.shape}')
print(f'注意力權重形狀: {attn.shape}')

### 24.2.3 MLP (Feed-Forward Network)

In [ ]:
class MLP(nn.Module):
    """
    Transformer 中的前饋網路
    
    兩層線性變換，中間使用 GELU 活化函數。
    """
    def __init__(self, embed_dim, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        hidden_dim = int(embed_dim * mlp_ratio)
        
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        """
        Args:
            x: (B, N, embed_dim)
        Returns:
            (B, N, embed_dim)
        """
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x

# 測試 MLP
mlp = MLP(embed_dim=768, mlp_ratio=4.0)
test_input = torch.randn(2, 197, 768)
output = mlp(test_input)
print(f'輸入形狀: {test_input.shape}')
print(f'輸出形狀: {output.shape}')
print(f'隱藏層維度: {int(768 * 4.0)}')

### 24.2.4 Transformer Block

In [ ]:
class TransformerBlock(nn.Module):
    """
    Transformer Encoder Block (Pre-LN 架構)
    
    結構:
    x -> LayerNorm -> Multi-Head Attention -> + (殘差) -> LayerNorm -> MLP -> + (殘差)
    """
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.0, drop_path=0.0):
        super().__init__()
        
        # Pre-LayerNorm
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, mlp_ratio, dropout)
        
        # DropPath (Stochastic Depth) 用於正則化
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
    
    def forward(self, x, return_attention=False):
        """
        Args:
            x: (B, N, embed_dim)
        Returns:
            (B, N, embed_dim)
        """
        if return_attention:
            attn_output, attn_weights = self.attn(self.norm1(x), return_attention=True)
            x = x + self.drop_path(attn_output)
            x = x + self.drop_path(self.mlp(self.norm2(x)))
            return x, attn_weights
        else:
            x = x + self.drop_path(self.attn(self.norm1(x)))
            x = x + self.drop_path(self.mlp(self.norm2(x)))
            return x

class DropPath(nn.Module):
    """
    Stochastic Depth: 訓練時隨機丟棄整個路徑
    """
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob
    
    def forward(self, x):
        if self.drop_prob == 0. or not self.training:
            return x
        
        keep_prob = 1 - self.drop_prob
        # 對每個樣本獨立決定是否丟棄
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor.floor_()  # 二值化
        output = x.div(keep_prob) * random_tensor
        return output

# 測試 Transformer Block
block = TransformerBlock(embed_dim=768, num_heads=12, mlp_ratio=4.0)
test_input = torch.randn(2, 197, 768)
output = block(test_input)
print(f'輸入形狀: {test_input.shape}')
print(f'輸出形狀: {output.shape}')

## 24.3 完整 Vision Transformer 模型

In [ ]:
class VisionTransformer(nn.Module):
    """
    Vision Transformer (ViT)
    
    將圖片分割成 patches，加上位置編碼和 CLS token，
    通過 Transformer Encoder 處理，最後用 CLS token 進行分類。
    """
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        in_channels=3,
        num_classes=1000,
        embed_dim=768,
        depth=12,
        num_heads=12,
        mlp_ratio=4.0,
        dropout=0.0,
        drop_path_rate=0.0
    ):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim
        self.num_patches = (img_size // patch_size) ** 2
        
        # Patch Embedding
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        
        # CLS Token: 可學習的分類標記
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        # Position Embedding: 可學習的位置編碼
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.pos_dropout = nn.Dropout(dropout)
        
        # Transformer Encoder
        # 使用線性增加的 drop path rate
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout, dpr[i])
            for i in range(depth)
        ])
        
        # Final LayerNorm
        self.norm = nn.LayerNorm(embed_dim)
        
        # Classification Head
        self.head = nn.Linear(embed_dim, num_classes)
        
        # 初始化權重
        self._init_weights()
    
    def _init_weights(self):
        # 初始化 CLS token 和位置編碼
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        
        # 初始化其他層
        self.apply(self._init_layer_weights)
    
    def _init_layer_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    
    def forward_features(self, x):
        """
        提取特徵（不包含分類頭）
        """
        B = x.shape[0]
        
        # Patch Embedding: (B, C, H, W) -> (B, num_patches, embed_dim)
        x = self.patch_embed(x)
        
        # 擴展 CLS token 到 batch 維度
        cls_tokens = self.cls_token.expand(B, -1, -1)  # (1, 1, D) -> (B, 1, D)
        
        # 連接 CLS token 和 patch embeddings
        x = torch.cat([cls_tokens, x], dim=1)  # (B, num_patches+1, D)
        
        # 加上位置編碼
        x = x + self.pos_embed
        x = self.pos_dropout(x)
        
        # 通過 Transformer Encoder
        for block in self.blocks:
            x = block(x)
        
        # Final LayerNorm
        x = self.norm(x)
        
        # 返回 CLS token 的輸出
        return x[:, 0]
    
    def forward(self, x):
        x = self.forward_features(x)
        x = self.head(x)
        return x
    
    def get_attention_maps(self, x):
        """
        獲取所有層的注意力圖
        """
        B = x.shape[0]
        attention_maps = []
        
        # Patch Embedding
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = x + self.pos_embed
        x = self.pos_dropout(x)
        
        # 收集每層的注意力
        for block in self.blocks:
            x, attn = block(x, return_attention=True)
            attention_maps.append(attn)
        
        return attention_maps

# 建立不同配置的 ViT
def vit_tiny(num_classes=1000, img_size=224):
    """ViT-Tiny: 用於快速測試"""
    return VisionTransformer(
        img_size=img_size, patch_size=16, embed_dim=192, depth=12,
        num_heads=3, num_classes=num_classes
    )

def vit_small(num_classes=1000, img_size=224):
    """ViT-Small"""
    return VisionTransformer(
        img_size=img_size, patch_size=16, embed_dim=384, depth=12,
        num_heads=6, num_classes=num_classes
    )

def vit_base(num_classes=1000, img_size=224):
    """ViT-Base: 原論文中的基準配置"""
    return VisionTransformer(
        img_size=img_size, patch_size=16, embed_dim=768, depth=12,
        num_heads=12, num_classes=num_classes
    )

def vit_large(num_classes=1000, img_size=224):
    """ViT-Large"""
    return VisionTransformer(
        img_size=img_size, patch_size=16, embed_dim=1024, depth=24,
        num_heads=16, num_classes=num_classes
    )

# 測試模型
print("建立 ViT-Tiny 模型...")
model = vit_tiny(num_classes=10, img_size=32)  # 適用於 CIFAR-10
test_input = torch.randn(2, 3, 32, 32)
output = model(test_input)
print(f'輸入形狀: {test_input.shape}')
print(f'輸出形狀: {output.shape}')
print(f'模型參數量: {sum(p.numel() for p in model.parameters()):,}')

## 24.4 模型配置比較

In [ ]:
def count_parameters(model):
    """計算模型參數量"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# 比較不同 ViT 配置
configs = {
    'ViT-Tiny': {'embed_dim': 192, 'depth': 12, 'num_heads': 3},
    'ViT-Small': {'embed_dim': 384, 'depth': 12, 'num_heads': 6},
    'ViT-Base': {'embed_dim': 768, 'depth': 12, 'num_heads': 12},
    'ViT-Large': {'embed_dim': 1024, 'depth': 24, 'num_heads': 16},
    'ViT-Huge': {'embed_dim': 1280, 'depth': 32, 'num_heads': 16},
}

results = []
for name, config in configs.items():
    model = VisionTransformer(
        img_size=224, patch_size=16, num_classes=1000, **config
    )
    params = count_parameters(model) / 1e6  # 轉換為百萬
    results.append({
        'Model': name,
        'Embed Dim': config['embed_dim'],
        'Depth': config['depth'],
        'Heads': config['num_heads'],
        'Params (M)': f'{params:.1f}'
    })
    print(f"{name}: {params:.1f}M 參數")

# 視覺化比較
fig, ax = plt.subplots(figsize=(10, 6))
models = [r['Model'] for r in results]
params = [float(r['Params (M)']) for r in results]

bars = ax.bar(models, params, color=['#2ecc71', '#3498db', '#9b59b6', '#e74c3c', '#f39c12'])
ax.set_ylabel('Parameters (Millions)', fontsize=12)
ax.set_xlabel('Model Variant', fontsize=12)
ax.set_title('Vision Transformer Model Sizes', fontsize=14)

# 在柱狀圖上方標註數值
for bar, param in zip(bars, params):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{param:.0f}M', ha='center', va='bottom', fontsize=10)

plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('vit_model_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

## 24.5 位置編碼視覺化

In [ ]:
def visualize_position_embedding(pos_embed, num_patches_per_side=14):
    """
    視覺化位置編碼的相似度矩陣
    
    Args:
        pos_embed: (1, N+1, D) 位置編碼
        num_patches_per_side: 每邊的 patch 數量
    """
    # 移除 CLS token 的位置編碼
    pos_embed = pos_embed[0, 1:, :]  # (N, D)
    
    # 計算餘弦相似度
    pos_embed_normalized = F.normalize(pos_embed, dim=-1)
    similarity = pos_embed_normalized @ pos_embed_normalized.T  # (N, N)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # 左圖：完整的相似度矩陣
    im1 = axes[0].imshow(similarity.detach().cpu().numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
    axes[0].set_xlabel('Patch Index', fontsize=11)
    axes[0].set_ylabel('Patch Index', fontsize=11)
    axes[0].set_title('Position Embedding Similarity Matrix', fontsize=12)
    plt.colorbar(im1, ax=axes[0], shrink=0.8)
    
    # 右圖：選取特定位置，顯示其與所有位置的相似度
    # 選擇中心位置
    center_idx = (num_patches_per_side // 2) * num_patches_per_side + num_patches_per_side // 2
    center_similarity = similarity[center_idx].reshape(num_patches_per_side, num_patches_per_side)
    
    im2 = axes[1].imshow(center_similarity.detach().cpu().numpy(), cmap='RdBu_r', vmin=-1, vmax=1)
    axes[1].set_xlabel('Patch Column', fontsize=11)
    axes[1].set_ylabel('Patch Row', fontsize=11)
    axes[1].set_title(f'Similarity with Center Patch (Index {center_idx})', fontsize=12)
    
    # 標記中心位置
    axes[1].scatter([num_patches_per_side // 2], [num_patches_per_side // 2], 
                    color='green', s=100, marker='x', linewidths=3)
    plt.colorbar(im2, ax=axes[1], shrink=0.8)
    
    plt.tight_layout()
    plt.savefig('position_embedding_similarity.png', dpi=150, bbox_inches='tight')
    plt.show()

# 建立一個訓練過的位置編碼模擬（使用隨機初始化）
# 實際應用中應使用訓練後的模型
model = vit_base(num_classes=1000, img_size=224)
print("位置編碼形狀:", model.pos_embed.shape)
visualize_position_embedding(model.pos_embed, num_patches_per_side=14)

## 24.6 注意力圖視覺化

In [ ]:
def visualize_attention_maps(model, image, num_patches_per_side=14):
    """
    視覺化不同層的注意力圖
    
    Args:
        model: ViT 模型
        image: (1, C, H, W) 輸入圖片
        num_patches_per_side: 每邊的 patch 數量
    """
    model.eval()
    with torch.no_grad():
        attention_maps = model.get_attention_maps(image)
    
    # 選擇要視覺化的層
    layers_to_show = [0, 3, 6, 11]  # 第 1, 4, 7, 12 層
    layers_to_show = [l for l in layers_to_show if l < len(attention_maps)]
    
    fig, axes = plt.subplots(2, len(layers_to_show), figsize=(4*len(layers_to_show), 8))
    
    for col, layer_idx in enumerate(layers_to_show):
        attn = attention_maps[layer_idx]  # (B, heads, N+1, N+1)
        
        # 取第一個樣本，平均所有注意力頭
        attn_mean = attn[0].mean(dim=0)  # (N+1, N+1)
        
        # 上排：CLS token 對所有 patches 的注意力
        cls_attn = attn_mean[0, 1:]  # 排除 CLS 對自己的注意力
        cls_attn = cls_attn.reshape(num_patches_per_side, num_patches_per_side)
        
        im1 = axes[0, col].imshow(cls_attn.cpu().numpy(), cmap='hot')
        axes[0, col].set_title(f'Layer {layer_idx + 1}: CLS Attention', fontsize=11)
        axes[0, col].axis('off')
        
        # 下排：中心 patch 對所有 patches 的注意力
        center_idx = (num_patches_per_side // 2) * num_patches_per_side + num_patches_per_side // 2
        center_attn = attn_mean[center_idx + 1, 1:]  # +1 是因為有 CLS token
        center_attn = center_attn.reshape(num_patches_per_side, num_patches_per_side)
        
        im2 = axes[1, col].imshow(center_attn.cpu().numpy(), cmap='hot')
        axes[1, col].set_title(f'Layer {layer_idx + 1}: Center Patch Attention', fontsize=11)
        axes[1, col].axis('off')
        
        # 標記中心位置
        axes[1, col].scatter([num_patches_per_side // 2], [num_patches_per_side // 2],
                            color='cyan', s=50, marker='x', linewidths=2)
    
    plt.suptitle('Attention Patterns Across Layers', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('attention_maps_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()

# 建立模型並視覺化注意力
model = VisionTransformer(
    img_size=224, patch_size=16, num_classes=1000,
    embed_dim=384, depth=12, num_heads=6
)

# 生成隨機測試圖片
test_image = torch.randn(1, 3, 224, 224)
visualize_attention_maps(model, test_image, num_patches_per_side=14)

## 24.7 在 CIFAR-10 上訓練小型 ViT

In [ ]:
# 資料增強和載入
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# 下載 CIFAR-10
print("載入 CIFAR-10 資料集...")
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"訓練集大小: {len(train_dataset)}")
print(f"測試集大小: {len(test_dataset)}")

In [ ]:
# 建立小型 ViT（適合 CIFAR-10 的 32x32 圖片）
class ViTForCIFAR(nn.Module):
    """
    適合 CIFAR-10 的小型 ViT
    
    由於 CIFAR-10 圖片只有 32x32，我們使用較小的 patch size
    """
    def __init__(self, num_classes=10):
        super().__init__()
        self.model = VisionTransformer(
            img_size=32,
            patch_size=4,      # 4x4 patches，產生 8x8=64 個 patches
            in_channels=3,
            num_classes=num_classes,
            embed_dim=256,
            depth=6,
            num_heads=8,
            mlp_ratio=4.0,
            dropout=0.1,
            drop_path_rate=0.1
        )
    
    def forward(self, x):
        return self.model(x)

model = ViTForCIFAR(num_classes=10).to(device)
print(f"模型參數量: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 訓練設定
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        
        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()
    
    return total_loss / len(loader), 100. * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            
            total_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    
    return total_loss / len(loader), 100. * correct / total

In [ ]:
# 訓練模型（減少 epoch 數以加快演示）
num_epochs = 20
train_losses, train_accs = [], []
test_losses, test_accs = [], []

print("開始訓練...")
for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    scheduler.step()
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}: '
              f'訓練損失={train_loss:.4f}, 訓練準確率={train_acc:.2f}% | '
              f'測試損失={test_loss:.4f}, 測試準確率={test_acc:.2f}%')

print(f"\n訓練完成！最終測試準確率: {test_accs[-1]:.2f}%")

In [ ]:
# 視覺化訓練過程
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 損失曲線
ax1.plot(train_losses, label='Training Loss', color='blue', linewidth=2)
ax1.plot(test_losses, label='Test Loss', color='red', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Test Loss', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# 準確率曲線
ax2.plot(train_accs, label='Training Accuracy', color='blue', linewidth=2)
ax2.plot(test_accs, label='Test Accuracy', color='red', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training and Test Accuracy', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('vit_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 24.8 ViT vs CNN 比較

In [ ]:
# 視覺化 ViT 和 CNN 在不同資料規模下的比較
# 基於論文中的實驗結果

data_sizes = ['ImageNet-1K\n(1.3M)', 'ImageNet-21K\n(14M)', 'JFT-300M\n(300M)']
resnet_acc = [77.6, 80.1, 87.0]  # ResNet-152x4
vit_acc = [64.7, 79.3, 88.6]     # ViT-H/14

x = np.arange(len(data_sizes))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

bars1 = ax.bar(x - width/2, resnet_acc, width, label='ResNet-152x4', color='#3498db')
bars2 = ax.bar(x + width/2, vit_acc, width, label='ViT-H/14', color='#e74c3c')

ax.set_ylabel('ImageNet Accuracy (%)', fontsize=12)
ax.set_xlabel('Pre-training Dataset', fontsize=12)
ax.set_title('ViT vs ResNet: Effect of Pre-training Data Scale', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(data_sizes, fontsize=10)
ax.legend(fontsize=11)
ax.set_ylim(60, 95)

# 標註數值
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=10)

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.5,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=10)

# 添加註解
ax.annotate('ViT needs large-scale\npre-training data', 
            xy=(0, 65), xytext=(0.5, 72),
            fontsize=10, color='gray',
            arrowprops=dict(arrowstyle='->', color='gray'))

ax.annotate('ViT surpasses CNN\nwith enough data', 
            xy=(2, 88.6), xytext=(1.5, 92),
            fontsize=10, color='gray',
            arrowprops=dict(arrowstyle='->', color='gray'))

ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('vit_vs_cnn_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 24.9 計算效率分析

In [ ]:
def analyze_computational_cost(img_sizes=[224, 384, 512], patch_size=16, embed_dim=768, depth=12):
    """
    分析不同輸入解析度下的計算成本
    """
    results = []
    
    for img_size in img_sizes:
        num_patches = (img_size // patch_size) ** 2
        
        # Self-attention 複雜度: O(N^2 * D)
        attn_flops = num_patches ** 2 * embed_dim * depth
        
        # MLP 複雜度: O(N * D^2 * 4)
        mlp_flops = num_patches * embed_dim * (4 * embed_dim) * 2 * depth
        
        total_flops = attn_flops + mlp_flops
        
        results.append({
            'img_size': img_size,
            'num_patches': num_patches,
            'attn_flops': attn_flops / 1e9,
            'mlp_flops': mlp_flops / 1e9,
            'total_flops': total_flops / 1e9
        })
    
    return results

results = analyze_computational_cost()

# 視覺化
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

img_sizes = [r['img_size'] for r in results]
num_patches = [r['num_patches'] for r in results]
attn_flops = [r['attn_flops'] for r in results]
mlp_flops = [r['mlp_flops'] for r in results]

# 左圖：Patch 數量
ax1.bar(range(len(img_sizes)), num_patches, color=['#3498db', '#e74c3c', '#2ecc71'])
ax1.set_xticks(range(len(img_sizes)))
ax1.set_xticklabels([f'{s}×{s}' for s in img_sizes])
ax1.set_xlabel('Input Resolution', fontsize=12)
ax1.set_ylabel('Number of Patches', fontsize=12)
ax1.set_title('Sequence Length vs Input Resolution', fontsize=14)
for i, v in enumerate(num_patches):
    ax1.text(i, v + 20, str(v), ha='center', fontsize=11)

# 右圖：計算量分解
x = np.arange(len(img_sizes))
width = 0.35

bars1 = ax2.bar(x - width/2, attn_flops, width, label='Self-Attention', color='#3498db')
bars2 = ax2.bar(x + width/2, mlp_flops, width, label='MLP', color='#e74c3c')

ax2.set_xticks(x)
ax2.set_xticklabels([f'{s}×{s}' for s in img_sizes])
ax2.set_xlabel('Input Resolution', fontsize=12)
ax2.set_ylabel('GFLOPs', fontsize=12)
ax2.set_title('Computational Cost Breakdown (ViT-Base)', fontsize=14)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.savefig('vit_computational_cost.png', dpi=150, bbox_inches='tight')
plt.show()

# 列印詳細資訊
print("\n計算成本分析 (ViT-Base, patch_size=16):")
print("="*60)
for r in results:
    print(f"解析度: {r['img_size']}×{r['img_size']}")
    print(f"  Patch 數量: {r['num_patches']}")
    print(f"  Self-Attention: {r['attn_flops']:.2f} GFLOPs")
    print(f"  MLP: {r['mlp_flops']:.2f} GFLOPs")
    print(f"  總計: {r['total_flops']:.2f} GFLOPs")
    print()

## 24.10 本章總結

### 主要實作內容

1. **Patch Embedding**: 將圖片分割成 patches 並進行線性投影
2. **Multi-Head Self-Attention**: 標準的 Transformer 注意力機制
3. **完整 ViT 模型**: 包含 CLS token、位置編碼和分類頭
4. **視覺化工具**: patch 分割、位置編碼相似度、注意力圖
5. **CIFAR-10 訓練範例**: 展示實際訓練流程

### 核心發現

- ViT 將圖片視為 patch 序列，完全拋棄卷積操作
- 在大規模資料上預訓練後，ViT 可以超越 CNN
- 注意力機制使模型能夠捕捉長距離依賴關係
- 位置編碼能夠學習到二維空間結構

### 實際應用建議

- 小資料集：使用預訓練模型微調，或使用混合架構
- 大資料集：從零訓練 ViT 可能更有優勢
- 高解析度：考慮使用 Swin Transformer 等高效變體

In [ ]:
print("第二十四章 Vision Transformer 實作完成！")
print("\n生成的圖片：")
print("- patch_visualization.png: Patch 分割視覺化")
print("- vit_model_sizes.png: 模型大小比較")
print("- position_embedding_similarity.png: 位置編碼相似度")
print("- attention_maps_visualization.png: 注意力圖視覺化")
print("- vit_training_curves.png: 訓練曲線")
print("- vit_vs_cnn_comparison.png: ViT vs CNN 比較")
print("- vit_computational_cost.png: 計算成本分析")